# Multi-Model LLM Engineering

Explore multiple LLM providers, local models, model routing, prompt caching, and multi-agent conversations using Python.

The notebook uses OpenAI-compatible APIs where available, alongside provider-specific SDKs and LiteLLM.

## Setup

Install the packages used in this notebook:

```bash
pip install openai anthropic google-genai python-dotenv litellm langchain-openai
```

Add the API keys for the providers you plan to use to a local `.env` file. Never commit `.env` or API keys to GitHub.

```text
OPENAI_API_KEY=
ANTHROPIC_API_KEY=
GOOGLE_API_KEY=
DEEPSEEK_API_KEY=
GROQ_API_KEY=
XAI_API_KEY=
OPENROUTER_API_KEY=
```

In [ ]:
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv()

def show_response(response):
    display(Markdown(response))

## 1. Connect to model providers

In [ ]:
# OpenAI
openai_client = OpenAI()

# OpenAI-compatible providers
gemini_client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)

groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

xai_client = OpenAI(
    api_key=os.getenv("XAI_API_KEY"),
    base_url="https://api.x.ai/v1",
)

openrouter_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

# Local Ollama server
ollama_client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1",
)

## 2. Compare model responses

Choose a model ID supported by the provider account you are using. Model availability and names change over time.

In [ ]:
messages = [
    {
        "role": "user",
        "content": "Tell a short joke for someone learning LLM engineering.",
    }
]

response = openai_client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
)

show_response(response.choices[0].message.content)

## 3. Reasoning and problem solving

In [ ]:
puzzle = [
    {
        "role": "user",
        "content": (
            "You toss two coins. At least one is heads. "
            "What is the probability the other coin is tails? "
            "Explain your reasoning."
        ),
    }
]

response = openai_client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=puzzle,
)

show_response(response.choices[0].message.content)

## 4. Test other providers

Run the examples for providers where you have configured an API key. Replace model IDs with models currently available to your account.

In [ ]:
# Google Gemini through its OpenAI-compatible endpoint
response = gemini_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
)
show_response(response.choices[0].message.content)

In [ ]:
# DeepSeek
response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
)
show_response(response.choices[0].message.content)

In [ ]:
# Groq
response = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages,
)
show_response(response.choices[0].message.content)

In [ ]:
# xAI
response = xai_client.chat.completions.create(
    model="grok-3-mini",
    messages=messages,
)
show_response(response.choices[0].message.content)

In [ ]:
# OpenRouter
response = openrouter_client.chat.completions.create(
    model="openai/gpt-4.1-mini",
    messages=messages,
)
show_response(response.choices[0].message.content)

## 5. Run a local model with Ollama

Start Ollama and pull a model before running these cells.

In [ ]:
# Run in a terminal first:
# ollama serve
# ollama pull llama3.2

response = ollama_client.chat.completions.create(
    model="llama3.2",
    messages=messages,
)
show_response(response.choices[0].message.content)

## 6. Use provider-specific SDKs

In [ ]:
from google import genai

gemini = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents="Describe the color blue in one sentence.",
)
print(response.text)

In [ ]:
from anthropic import Anthropic

anthropic = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

response = anthropic.messages.create(
    model="claude-3-5-haiku-latest",
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": "Describe the color blue in one sentence.",
        }
    ],
)
print(response.content[0].text)

## 7. Route requests with LiteLLM

LiteLLM provides a common interface for calling models from different providers.

In [ ]:
from litellm import completion

response = completion(
    model="openai/gpt-4.1-mini",
    messages=messages,
)

show_response(response.choices[0].message.content)

if response.usage:
    print("Input tokens:", response.usage.prompt_tokens)
    print("Output tokens:", response.usage.completion_tokens)
    print("Total tokens:", response.usage.total_tokens)

## 8. Prompt caching

Prompt caching can reduce latency and input-token costs when supported by the model provider. Cache behavior, eligibility, and pricing differ by provider.

For the best chance of a cache hit:
- Put stable instructions and shared context at the beginning of the prompt.
- Put changing user-specific content near the end.
- Reuse an identical prompt prefix.

Provider documentation:
- [OpenAI prompt caching](https://platform.openai.com/docs/guides/prompt-caching)
- [Anthropic prompt caching](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching)
- [Gemini context caching](https://ai.google.dev/gemini-api/docs/caching)

## 9. Build a multi-agent conversation

In [ ]:
from openai import OpenAI

client = OpenAI()

messages = [
    {
        "role": "system",
        "content": (
            "You are a thoughtful assistant. Challenge ideas respectfully "
            "and explain your reasoning."
        ),
    },
    {
        "role": "user",
        "content": "Is it better to learn Python before studying LLMs?",
    },
]

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
)

assistant_reply = response.choices[0].message.content
show_response(assistant_reply)

### Continue the conversation with history

The messages list carries the conversation context between API calls.

In [ ]:
messages.append(
    {"role": "assistant", "content": assistant_reply}
)
messages.append(
    {
        "role": "user",
        "content": "What would be a practical learning path?",
    }
)

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
)

show_response(response.choices[0].message.content)

## Key takeaways

- Connect to hosted and locally running language models.
- Use OpenAI-compatible APIs and provider-specific SDKs.
- Route model requests through LiteLLM.
- Understand prompt caching and token usage.
- Maintain conversation history across requests.

**Note:** Model IDs and provider features change. Check the provider's current documentation before running an example.